In [1]:
from dotenv import load_dotenv
_ = load_dotenv()

import time
import openai
from openai import OpenAI

client = OpenAI()

In [2]:
# ----------------------------
# 1. Create a temporary file
# ----------------------------
DOCUMENT_TEXT = """\
The Morava Virus is a fictional virus described in this document.
It causes a disease known as Morava Hemorrhagic Fever.
Transmission occurs through contact with infected bodily fluids.
Symptoms include fever, vomiting, muscle pain, and bleeding.
There is currently no treatment or vaccine for this disease.
"""

with open("morava_virus.txt", "w") as f:
    f.write(DOCUMENT_TEXT)

print("Created local file: morava_virus.txt")


Created local file: morava_virus.txt


In [3]:
# ----------------------------
# 2. Upload file
# ----------------------------
file_obj = client.files.create(file=open("morava_virus.txt", "rb"), purpose="assistants")
file_id = file_obj.id
print(f"Uploaded file ID: {file_id}")


Uploaded file ID: file-AqvCj7p5MPqAxEsQgxzPCd


In [4]:
# ----------------------------
# 3. Create vector store and attach file
# ----------------------------
vector_store = client.vector_stores.create(name="MoravaVirusDemo")
vector_store_id = vector_store.id
print(f"Created vector store ID: {vector_store_id}")

client.vector_stores.file_batches.create(
    vector_store_id=vector_store_id,
    file_ids=[file_id],
)
print("Added file to vector store.")


Created vector store ID: vs_68fba9504a748191ba981df3bb3697be
Added file to vector store.


In [5]:
# ----------------------------
# 4. Wait for indexing to finish
# ----------------------------
def wait_for_vector_store(client, vector_store_id, timeout=120, interval=5):
    print("Waiting for vector store files to finish processing...")
    start = time.time()
    while time.time() - start < timeout:
        vs = client.vector_stores.retrieve(vector_store_id)
        counts = vs.file_counts
        if counts.completed == counts.total:
            print("✅ Vector store is ready.")
            return
        time.sleep(interval)
        print("Still processing...")
    raise TimeoutError("Vector store did not become ready in time.")

wait_for_vector_store(client, vector_store_id)


Waiting for vector store files to finish processing...
✅ Vector store is ready.


In [6]:
# ----------------------------
# 5. Strict factual prompt
# ----------------------------
SYSTEM_PROMPT = """\
You are a factual assistant. 
You must never speculate or assume information.
If you don't find information explicitly stated in the uploaded files or in reliable knowledge, respond clearly that you have no verified information.

Rules:
- If the virus or disease is not mentioned in the provided document, say so plainly.
- Do not guess, infer, or fill in gaps.
- If asked about something fictitious not found in the text, respond:
  "I have no verified information about that."

Example:
Q: What is the Morava Virus?
A: I have no verified information about that.
"""


In [7]:
# ----------------------------
# 6. Questions
# ----------------------------
questions = [
    "What is the Morava Virus and the disease it causes?",
    "How is the Morava Virus transmitted between humans?",
    "What are the symptoms of Morava Hemorrhagic Fever?",
    "Is there a treatment or vaccine for Morava Hemorrhagic Fever?",
]

def ask_questions(client, questions, vector_store_id=None):
    phase = "PHASE 2: Asking WITH vector store" if vector_store_id else "PHASE 1: Asking WITHOUT background file"
    print(f"\n=== {phase} ===\n")

    tools = []
    if vector_store_id:
        tools = [{"type": "file_search", "vector_store_ids": [vector_store_id]}]

    for q in questions:
        response = client.responses.create(
            model="gpt-4.1",
            input=[{"role": "system", "content": SYSTEM_PROMPT},
                   {"role": "user", "content": q}],
            tools=tools,
        )
        answer = response.output_text.strip()
        print(f"Q: {q}\nA: {answer}\n{'-'*60}")


In [8]:
# ----------------------------
# 7. Run both phases
# ----------------------------
ask_questions(client, questions)  # Phase 1: no background file
ask_questions(client, questions, vector_store_id)  # Phase 2: with file_search

print("\n✅ Demo complete.")



=== PHASE 1: Asking WITHOUT background file ===

Q: What is the Morava Virus and the disease it causes?
A: I have no verified information about that.
------------------------------------------------------------
Q: How is the Morava Virus transmitted between humans?
A: I have no verified information about that.
------------------------------------------------------------
Q: What are the symptoms of Morava Hemorrhagic Fever?
A: I have no verified information about that.
------------------------------------------------------------
Q: Is there a treatment or vaccine for Morava Hemorrhagic Fever?
A: I have no verified information about that.
------------------------------------------------------------

=== PHASE 2: Asking WITH vector store ===

Q: What is the Morava Virus and the disease it causes?
A: The Morava Virus is a fictional virus that causes a disease known as Morava Hemorrhagic Fever. Transmission occurs through contact with infected bodily fluids. Symptoms of the disease include

In [9]:

# How to delete a file_batch?
# client.vector_stores.file_batches.create(
# How do they differ from just files?
# Or it is just a way to bulk upload?


print(client.vector_stores.files.delete(
    vector_store_id=vector_store_id,
    file_id=file_id
))

print(client.files.delete(file_id))

print(client.vector_stores.delete(vector_store_id=vector_store_id))


VectorStoreFileDeleted(id='file-AqvCj7p5MPqAxEsQgxzPCd', deleted=True, object='vector_store.file.deleted')
FileDeleted(id='file-AqvCj7p5MPqAxEsQgxzPCd', deleted=True, object='file')
VectorStoreDeleted(id='vs_68fba9504a748191ba981df3bb3697be', deleted=True, object='vector_store.deleted')
